
# EDA – Dataset de Limpeza de Hub

Autor:  Matheus Santos

Atualizado: 19/02/2026

## Objetivo:
Realizar a análise exploratória de dados do dataset de limpeza de hub.


In [ ]:
!pip install nbconvert

## Imports necessários

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
import hashlib
from math import ceil
import matplotlib.pyplot as plt

## Constantes

In [ ]:
# CONFIGURAÇÃO DO DATASET
DATASET_DIR = Path("C:\\Users\\GHPM\\Downloads\\hubDetectV1\\hubDetectV1")
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

IMG_EXTS = [".jpg", ".jpeg", ".png"]


## Funções

In [ ]:
# =========================
# Image hashing and utility functions
# =========================

def file_md5(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute the MD5 hash of a file (exact duplicate check).

    Args:
        path: Path to the file.
        chunk_size: Read file in chunks to handle large files.

    Returns:
        Hexadecimal MD5 hash string.
    """
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def dhash_bgr(img_bgr: np.ndarray, hash_size: int = 8) -> int:
    """Compute dHash (perceptual hash, robust to small changes).

    Args:
        img_bgr: Image in BGR format (OpenCV default).
        hash_size: Size of hash (default 8). Returns hash_size*hash_size bits.

    Returns:
        Integer representing the perceptual hash.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (hash_size + 1, hash_size), interpolation=cv2.INTER_AREA)
    diff = resized[:, 1:] > resized[:, :-1]
    bits = diff.flatten().astype(np.uint8)
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out

def hamming_distance_int(a: int, b: int) -> int:
    """
    Compute the Hamming distance between two integer values.

    The Hamming distance is defined as the number of bit positions at which
    the corresponding bits of two values are different. This implementation
    uses the bitwise XOR operation to identify differing bits and then counts
    the number of set bits (1s) in the result.

    Parameters
    ----------
    a : int
        First integer value (e.g., a hash or fingerprint).
    b : int
        Second integer value to compare against.

    Returns
    -------
    int
        The number of differing bit positions between `a` and `b`.

    """
    # XOR identifies the bit positions where a and b differ:
    # bits that are different become 1, equal bits become 0.
    xor_result = a ^ b

    # Count the number of 1s in the XOR result,
    # which equals the Hamming distance.
    return xor_result.bit_count()

def read_bgr_safe(path: Path) -> np.ndarray | None:
    """Read an image safely in BGR format using OpenCV.

    Returns:
        Image array or None if reading failed.
    """
    return cv2.imread(str(path))


def find_image_by_stem(stem: str) -> Path | None:
    """Find an image by its stem (filename without extension) inside IMAGES_DIR.

    Args:
        stem: Base filename without extension.

    Returns:
        Full Path if found, else None.
    """
    for ext in IMG_EXTS:
        p = IMAGES_DIR / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def safe_read_rgb(img_path: Path) -> np.ndarray | None:
    """Read an image and return it in RGB format. Returns None if failed."""
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        return None
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def draw_boxes_on_rgb(rgb: np.ndarray, labels_df: pd.DataFrame) -> np.ndarray:
    """Draw bounding boxes on an RGB image using a labels DataFrame.

    Args:
        rgb: RGB image.
        labels_df: DataFrame containing bounding box annotations.

    Returns:
        Image with boxes drawn in RGB format.
    """
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    bgr = draw_boxes(bgr, labels_df)  # Use existing draw_boxes function
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def show_images_grid(items, title, cols=4, figsize_scale=4):
    """Display a grid of images with optional labels and titles.

    Args:
        items: List of dictionaries containing:
            - "rgb" (np.ndarray) OR "image_path" (str/Path) + optional "stem"
            - "title" (str)
            - "labels_df" (pd.DataFrame, optional) for drawing boxes
        title: Overall title for the grid.
        cols: Number of columns in the grid.
        figsize_scale: Scaling factor for figure size.
    """
    n = len(items)
    if n == 0:
        print(f"{title}: nothing to display.")
        return

    rows = ceil(n / cols)
    plt.figure(figsize=(figsize_scale * cols, figsize_scale * rows))

    for i, it in enumerate(items, 1):
        plt.subplot(rows, cols, i)

        rgb = it.get("rgb", None)
        if rgb is None:
            p = Path(it["image_path"])
            rgb = safe_read_rgb(p)

        if rgb is None:
            plt.text(0.5, 0.5, "Failed to read\nimage", ha="center", va="center")
            plt.axis("off")
            plt.title(it.get("title", ""), fontsize=9)
            continue

        labels_df = it.get("labels_df", None)
        if labels_df is not None and len(labels_df) > 0:
            rgb = draw_boxes_on_rgb(rgb, labels_df)

        plt.imshow(rgb)
        plt.axis("off")
        plt.title(it.get("title", ""), fontsize=9)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


# =========================
# YOLO bounding box utilities
# =========================

def yolo_to_xyxy(xc, yc, bw, bh, W, H):
    """Convert YOLO normalized coordinates to absolute (x1, y1, x2, y2)."""
    x1 = int((xc - bw / 2) * W)
    y1 = int((yc - bh / 2) * H)
    x2 = int((xc + bw / 2) * W)
    y2 = int((yc + bh / 2) * H)
    return x1, y1, x2, y2


def draw_invalid_boxes(img_rgb: np.ndarray, df_img: pd.DataFrame) -> np.ndarray:
    """Draw invalid bounding boxes in red on an RGB image.

    Args:
        img_rgb: RGB image.
        df_img: DataFrame with columns: class, x, y, w, h.

    Returns:
        Image with invalid boxes drawn.
    """
    H, W = img_rgb.shape[:2]
    out = img_rgb.copy()

    for _, r in df_img.iterrows():
        cls = int(r["class"])
        xc, yc, bw, bh = float(r["x"]), float(r["y"]), float(r["w"]), float(r["h"])
        x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, bw, bh, W, H)

        # Clip coordinates for drawing without breaking
        x1c, y1c = max(0, x1), max(0, y1)
        x2c, y2c = min(W - 1, x2), min(H - 1, y2)

        # Correct inverted coordinates if any
        if x2c < x1c:
            x1c, x2c = x2c, x1c
        if y2c < y1c:
            y1c, y2c = y2c, y1c

        cv2.rectangle(out, (x1c, y1c), (x2c, y2c), (255, 0, 0), 2)
        cv2.putText(out, f"cls:{cls}", (x1c, max(15, y1c - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    return out


def draw_boxes(img: np.ndarray, labels_df: pd.DataFrame) -> np.ndarray:
    """Draw bounding boxes in green on an image (BGR format).

    Args:
        img: Image in BGR format.
        labels_df: DataFrame with normalized YOLO boxes (x, y, w, h) and class.

    Returns:
        Image with bounding boxes drawn.
    """
    h, w, _ = img.shape
    for _, r in labels_df.iterrows():
        xc, yc, bw, bh = r["x"], r["y"], r["w"], r["h"]
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, str(r["class"]), (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img


## Estatísticas do Dataset

#### Esta seção lista todos os arquivos de imagem em IMAGES_DIR e todos os arquivos de rótulo em LABELS_DIR. Apenas arquivos com extensões permitidas (IMG_EXTS) são considerados como imagens e apenas arquivos com extensão “.txt” são considerados como rótulos.

In [ ]:
# =========================
# List image and label files
# =========================

# List all image files with allowed extensions
images = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMG_EXTS]

# List all label files (.txt) in the labels directory
labels = list(LABELS_DIR.glob("*.txt"))

# Print counts
print(f"Images: {len(images)}")
print(f"Labels: {len(labels)}")

### Esta seção verifica inconsistências entre imagens e arquivos de rótulo comparando seus nomes base (nome do arquivo sem a extensão).

- Ela identifica:

    - imagens que não possuem um rótulo correspondente
    - rótulos que não possuem uma imagem correspondente

In [ ]:
# =========================
# Match images with labels
# =========================

# Extract stems (filenames without extension) for images and labels
image_stems = {p.stem for p in images}
label_stems = {p.stem for p in labels}

# Identify images that have no label
imgs_sem_label = image_stems - label_stems

# Identify labels that have no corresponding image
labels_sem_img = label_stems - image_stems

# Print counts of mismatches
print(f"Images without labels: {len(imgs_sem_label)}")
print(f"Labels without images: {len(labels_sem_img)}")


### Esta seção identifica nomes base de imagens (nome do arquivo sem a extensão) que aparecem em mais de um arquivo, o que pode indicar duplicatas ou múltiplas versões da mesma imagem.

Passos:

1. Extrair os nomes base (stems) de todas as imagens.
2. Contar as ocorrências de cada nome base.
3. Identificar os nomes base que aparecem em mais de uma imagem.
4. Mapear os nomes base para seus nomes de arquivos completos e exibir exemplos.

In [ ]:
# =========================
# Detect duplicate image stems
# =========================

# Extract stems (filenames without extension) for all images
image_stems = [p.stem for p in images]

# Count occurrences of each stem
img_stem_counts = Counter(image_stems)

# Identify stems that have more than one image
multi_image_stems = [stem for stem, cnt in img_stem_counts.items() if cnt > 1]
print(f"Stems with more than one image: {len(multi_image_stems)}")

# Map stem -> list of filenames (with extensions)
stem_to_files = defaultdict(list)
for p in images:
    stem_to_files[p.stem].append(p.name)

# Show some examples of stems with multiple images
for stem in multi_image_stems[:10]:  # display first 10 examples
    files = sorted(stem_to_files[stem])
    print(f"\n{stem}: {len(files)} images")
    for fn in files:
        print(f"  - {fn}")


In [ ]:
# =========================
# Load YOLO labels into a DataFrame
# =========================

records = []

# Iterate through each label file
for lbl in labels:
    with open(lbl, "r") as f:
        for line in f:
            # Parse line: class x y w h
            c, x, y, w, h = map(float, line.split())
            records.append({
                "image": lbl.stem,     # image filename stem
                "class": int(c),       # object class as integer
                "x": x, "y": y, "w": w, "h": h
            })

# Convert the list of records into a DataFrame
df = pd.DataFrame(records)

# Display first 5 rows
df.head(5)


### Esta seção calcula e visualiza a distribuição das classes de objetos no conjunto de dados com base no DataFrame de rótulos no formato YOLO.

In [ ]:
# =========================
# Analyze class distribution
# =========================

# Count number of instances per class, sorted by class index
class_counts = df["class"].value_counts().sort_index()

# Display counts in a table
display(class_counts)

# Plot class distribution as a bar chart
class_counts.plot(kind="bar", title="Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of instances")
plt.show()


In [ ]:
objs_per_img = df.groupby("image").size()
objs_per_img.describe()

In [ ]:
objs_per_img.hist(bins=20)
plt.title("Objetos por Imagem")
plt.show()


### Esta seção detecta imagens duplicadas exatas calculando o hash MD5 de cada arquivo de imagem. Também registra as imagens que não puderam ser lidas.

In [ ]:
# =========================
# 1A) Exact duplicate detection using MD5
# =========================

md5_rows = []
bad_reads = []

for p in images:
    if not Path(p).exists():
        bad_reads.append((str(p), "path_missing"))
        continue
    try:
        md5_rows.append({
            "image_path": str(p),
            "image_file": Path(p).name,
            "md5": file_md5(Path(p))  # compute exact MD5 hash
        })
    except Exception as e:
        bad_reads.append((str(p), f"md5_error: {e}"))

# Convert to DataFrame
df_md5 = pd.DataFrame(md5_rows)

# Group by hash to find duplicates
dup_md5_groups = (
    df_md5.groupby("md5")
          .agg(n=("image_file", "count"),
               files=("image_file", list),
               paths=("image_path", list))
          .reset_index()
          .query("n > 1")  # only keep duplicates
          .sort_values("n", ascending=False)
)

print(f"Exact duplicates (MD5): {len(dup_md5_groups)} groups")
display(dup_md5_groups[["n","files"]].head(20))

if bad_reads:
    print(f"Problems reading {len(bad_reads)} images (missing/corrupted). Examples:")
    for x in bad_reads[:10]:
        print(" -", x)


# =========================
# 1B) Perceptual duplicates using dHash
# =========================

ph_rows = []

for p in images:
    p = Path(p)
    bgr = read_bgr_safe(p)
    if bgr is None:
        continue
    dh = dhash_bgr(bgr, hash_size=8)  # 64-bit perceptual hash
    bucket = dh >> 48  # top 16 bits as bucket for efficiency
    ph_rows.append({
        "image_path": str(p),
        "image_file": p.name,
        "dhash": dh,
        "bucket": bucket
    })

df_ph = pd.DataFrame(ph_rows)
print("Images with perceptual hash calculated:", len(df_ph))

# Hamming distance threshold for similarity (smaller = stricter)
HAMMING_THRESHOLD = 5

near_dups = []

# Compare images within each bucket
for bucket, g in df_ph.groupby("bucket"):
    items = g.to_dict("records")
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            d = hamming_distance_int(int(items[i]["dhash"]), int(items[j]["dhash"]))
            if d <= HAMMING_THRESHOLD:
                near_dups.append({
                    "image_a": items[i]["image_file"],
                    "image_b": items[j]["image_file"],
                    "dist": d,
                    "path_a": items[i]["image_path"],
                    "path_b": items[j]["image_path"],
                    "bucket": bucket
                })

df_near_dups = pd.DataFrame(near_dups).sort_values(["dist","bucket"]).reset_index(drop=True)

print(f"Visually similar image pairs (dHash <= {HAMMING_THRESHOLD}): {len(df_near_dups)}")
display(df_near_dups.head(50))


# =========================
# 1C) Duplicate bounding boxes within the same image
# =========================

df_lbl = df.copy()
ROUND_N = 6  # decimal rounding for comparison

for c in ["x", "y", "w", "h"]:
    df_lbl[c] = df_lbl[c].round(ROUND_N)

# Group by image, class, and box coordinates to find duplicates
dup_lbl = (
    df_lbl.groupby(["image","class","x","y","w","h"])
          .size()
          .reset_index(name="count")
          .query("count > 1")  # only duplicates
          .sort_values("count", ascending=False)
)

print(f"Duplicated boxes (same class & coordinates, rounded to {ROUND_N}): {len(dup_lbl)} occurrences")
display(dup_lbl.head(30))

# Summary: total duplicated boxes per image
dup_lbl_by_img = (
    dup_lbl.groupby("image")["count"]
           .sum()
           .sort_values(ascending=False)
           .reset_index(name="n_dup_boxes")
)
display(dup_lbl_by_img.head(30))


### Esta seção visualiza imagens duplicadas exatas detectadas pelo hash MD5.

Para cada grupo de duplicatas:

- Exibe até max_per_group imagens
- Mostra os rótulos correspondentes (se disponíveis)
- Limita o número total de grupos exibidos a max_groups

In [ ]:
# =========================
# Display exact duplicate image groups (MD5)
# =========================

max_groups = 3      # Maximum number of MD5 duplicate groups to display
max_per_group = 8   # Maximum number of images per group

for gi, row in dup_md5_groups.head(max_groups).iterrows():
    # Take first `max_per_group` paths for this duplicate group
    paths = row["paths"][:max_per_group]
    items = []

    for p in paths:
        p = Path(p)
        stem = p.stem

        # Get label boxes for this image from the DataFrame
        lbls = df[df["image"] == stem]

        # Prepare dict for grid display
        items.append({
            "image_path": p,
            "labels_df": lbls,
            "title": f"{p.name}\n{stem}.txt"
        })

    # Display images in a grid with labels overlaid
    show_images_grid(
        items,
        title=f"Exact duplicates (MD5) – group with n={row['n']}",
        cols=4
    )


### Esta seção visualiza os top_k_pairs de imagens visualmente semelhantes detectadas usando hash perceptual (dHash).

Para cada par:

- Exibe ambas as imagens lado a lado
- Mostra os rótulos correspondentes (se disponíveis)
- Inclui a distância de Hamming no título

In [ ]:
# =========================
# Display visually similar image pairs (dHash)
# =========================

top_k_pairs = 10  # number of top similar pairs to display

# Take top-k closest pairs (smallest Hamming distance)
pairs = df_near_dups.head(top_k_pairs).to_dict("records")
items = []

for r in pairs:
    pa = Path(r["path_a"])
    pb = Path(r["path_b"])
    stem_a, stem_b = pa.stem, pb.stem

    # Get label boxes for each image
    lbl_a = df[df["image"] == stem_a]
    lbl_b = df[df["image"] == stem_b]

    # Prepare items for grid display
    items.append({
        "image_path": pa,
        "labels_df": lbl_a,
        "title": f"A: {pa.name}\n{stem_a}.txt | d={r['dist']}"
    })
    items.append({
        "image_path": pb,
        "labels_df": lbl_b,
        "title": f"B: {pb.name}\n{stem_b}.txt | d={r['dist']}"
    })

# Display images in a grid (2 columns: each pair side by side)
show_images_grid(
    items,
    title=f"Visually similar images – top {top_k_pairs} pairs",
    cols=2
)


### Esta seção detecta caixas delimitadoras (bounding boxes) com coordenadas fora do intervalo válido [0, 1]. Ela gera um DataFrame resumindo:

- imagens com caixas inválidas
- arquivos de rótulo correspondentes
- número de caixas inválidas por imagem

Também exibe uma grade de imagens com as caixas inválidas destacadas em vermelho.

In [ ]:
mask_invalid = (
    (df[["x","y","w","h"]] < 0).any(axis=1) |
    (df[["x","y","w","h"]] > 1).any(axis=1)
)

# Subset of invalid bounding boxes
invalid_boxes = df[mask_invalid].copy()
print(f"Invalid bounding boxes (rows): {len(invalid_boxes)}")

# List of image stems containing invalid boxes
invalid_images = sorted(invalid_boxes["image"].unique().tolist())

# -------------------------
# Build DataFrame linking images and label files
# -------------------------
rows = []

for stem in invalid_images:
    label_path = LABELS_DIR / f"{stem}.txt"

    # Try to find corresponding image file in known extensions
    img_path = None
    for ext in IMG_EXTS:
        candidate = IMAGES_DIR / f"{stem}{ext}"
        if candidate.exists():
            img_path = candidate
            break

    rows.append({
        "image_stem": stem,
        "image_file": img_path.name if img_path else None,
        "label_file": label_path.name if label_path.exists() else None,
        "image_path": str(img_path) if img_path else None,
        "label_path": str(label_path) if label_path.exists() else None,
        "n_invalid_boxes": int((invalid_boxes["image"] == stem).sum()),
    })

# Create DataFrame summarizing invalid files
df_invalid_files = pd.DataFrame(rows).sort_values(
    ["n_invalid_boxes", "image_stem"], ascending=[False, True]
)

# Display summary table
display(df_invalid_files[["image_file","label_file","n_invalid_boxes"]])

# -------------------------
# Visualize invalid bounding boxes
# -------------------------
if len(df_invalid_files) == 0:
    print("No invalid bounding boxes found.")
else:
    n_show = min(12, len(df_invalid_files))   # number of images to display
    cols = 4
    rows_n = ceil(n_show / cols)

    plt.figure(figsize=(4*cols, 4*rows_n))

    for i in range(n_show):
        rec = df_invalid_files.iloc[i]
        stem = rec["image_stem"]
        img_path = rec["image_path"]

        plt.subplot(rows_n, cols, i+1)

        # Handle missing image files
        if img_path is None:
            plt.text(0.5, 0.5, f"Image not found\n{rec['image_file']}",
                     ha="center", va="center")
            plt.axis("off")
            continue

        # Read image (BGR -> RGB)
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # Filter only invalid boxes for this image
        df_img_invalid = invalid_boxes[invalid_boxes["image"] == stem]

        # Draw invalid boxes in red
        img_draw = draw_invalid_boxes(img, df_img_invalid)

        # Display
        plt.imshow(img_draw)
        plt.axis("off")
        plt.title(f"{rec['image_file']}\n{rec['label_file']} | inv:{rec['n_invalid_boxes']}", fontsize=9)

    plt.tight_layout()
    plt.show()


### Esta seção visualiza uma grade de imagens válidas do conjunto de dados com suas caixas delimitadoras (bounding boxes) desenhadas. Apenas as imagens que foram lidas com sucesso são exibidas.

In [ ]:
# =========================
# Display a grid of valid images with bounding boxes
# =========================

n_show = 12       # number of images to show
rows, cols = 3, 4 # grid layout
plt.figure(figsize=(4*cols, 4*rows))

shown = 0
i_plot = 1
bad = []  # list to track images that could not be read

for img_path in images:  # iterate over all images until we reach n_show
    if shown >= n_show:
        break

    # Read image using OpenCV (BGR format)
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        bad.append(img_path)
        continue

    # Convert to RGB for matplotlib display
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Get bounding boxes for this image
    lbls = df[df["image"] == img_path.stem]

    # Draw bounding boxes in green
    img = draw_boxes(img, lbls)

    # Plot image in the grid
    plt.subplot(rows, cols, i_plot)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name, fontsize=9)

    shown += 1
    i_plot += 1

plt.tight_layout()
plt.show()

# -------------------------
# Report images that could not be read
# -------------------------
if bad:
    print(f"\n{len(bad)} images could not be read by OpenCV (cv2.imread returned None). Examples:")
    for p in bad[:20]:
        print(" -", p)


In [ ]:
!jupyter nbconvert --to html eda_limpeza_hub.ipynb